In [2]:
!pip install torch

  Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached cuda_bindings-13.3.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.5 kB)
  Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.29.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
  Using cached triton-3.7.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
  Using

In [11]:
import torch
%matplotlib inline
import hist
from hist import Hist
import coffea.processor as processor
import awkward as ak
from coffea.nanoevents import schemas

In [15]:
class Processor(processor.ProcessorABC):
    def __init__(self):
        dataset_axis = hist.cat('dataset','')
        MET_axis = hist.bin("MET","MET [GeV]", 50, 0, 100)

        self._accumulator = processor.dict_accumulator({'MET': hist.Hist("Counts", dataset_axis, MET_axis),
                                                       'Cutflow': processor.defaultdict_accumulator(int)})
    @property
    def accumulator(self):
        return self._accumulator

    def process(self, events):
        output = self.accumulator.identity()

        dataset = events.metadata["dataset"]
        MET = events.MET.pt

        output['cutflow']['all events'] += ak.size(MET)
        output['cutflow']['number of chunks'] += 1
        output['MET'].fill(dataset=dataset, MET=MET)

        return output

    def postprocess(self, accumulator):
        return accumulator

In [16]:
from dask.distributed import Client

client = Client("tls://localhost:8786")

In [17]:
fileset = {'SingleMu' : ["root://eospublic.cern.ch//eos/root-eos/benchmark/Run2012B_SingleMu.root"]}

executor = processor.DaskExecutor(client=client)

run = processor.Runner(executor=executor,
                        schema=schemas.NanoAODSchema,
                        savemetrics=True
                      )

output, metrics = run(fileset, "Events", processor_instance=Processor())

metrics

AttributeError: module hist has no attribute cat